# Calenda — Qwen3-0.6B LoRA 학습 (Colab · 단일 GPU)

단일 베이스 **Qwen/Qwen3-0.6B** · **completion-only 손실 ON** (프롬프트 마스킹, 응답 토큰만 학습).

**GPU 우선순위:** Blackwell(RTX PRO 6000 등) / H100 / A100 → 잡히면 그대로 사용.  
L4도 무방. **T4는 bf16 미지원(fp16 강제 → 품질 손실)이라 비권장.**

데이터·코드는 repo에 포함 (별도 업로드 불필요). 위→아래 순서 실행.  
(선택) 좌측 🔑 보안 비밀에 `HF_TOKEN` 등록 시 HuggingFace 인증 다운로드.

## 1. GPU 확인

In [ ]:
!nvidia-smi

## 2. repo clone + 버전 배너

In [ ]:
import os
%cd /content
if not os.path.exists('Calenda'):
    !git clone https://github.com/sooryong/Calenda.git
else:
    !cd Calenda && git fetch origin && git reset --hard origin/main
%cd /content/Calenda
import yaml, json, subprocess
from collections import Counter
_c = yaml.safe_load(open('configs/train_qwen3_0_6b.yaml', encoding='utf-8'))
def _stat(p):
    rows=[json.loads(l) for l in open(p,encoding='utf-8') if l.strip()]
    c=Counter(str(r['gold'].get('schedule_status')) for r in rows)
    return len(rows), c.get('yes',0), c.get('pending',0), c.get('no',0)
_tt,_ty,_tp,_tn=_stat('data/processed/train.jsonl')
_gt,_gy,_gp,_gn=_stat('data/eval/golden.jsonl')
_head=subprocess.run('git log --oneline -1',shell=True,capture_output=True,text=True).stdout.strip()
print('='*62)
print(f"  라운드          : {_c['run_name']}  ·  epochs={_c['num_train_epochs']}")
print(f"  completion-only : {_c.get('completion_only_loss', False)}   (#1: 프롬프트 마스킹, 응답만 학습)")
print(f"  커밋            : {_head[:60]}")
print(f"  학습셋          : train {_tt}   yes {_ty} / pending {_tp} / no {_tn}")
print(f"  평가셋          : golden {_gt}   yes {_gy} / pending {_gp} / no {_gn}")
print('='*62)

## 3. 라운드 설정 (CONFIG)

In [ ]:
import os
try:
    from google.colab import userdata
    _hf = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = os.environ['HUGGING_FACE_HUB_TOKEN'] = _hf
    print('  HF_TOKEN : 설정됨')
except Exception:
    print('  HF_TOKEN : 없음 — 익명 다운로드(공개 모델이라 동작)')
_REPO = '/content/Calenda'
if not os.path.isfile('configs/train_qwen3_0_6b.yaml') and os.path.isdir(_REPO):
    os.chdir(_REPO)
assert os.path.isfile('configs/train_qwen3_0_6b.yaml'), f'레포 루트 아님(cwd={os.getcwd()}). 2번 clone 셀 먼저.'
CONFIG = 'configs/train_qwen3_0_6b.yaml'
import yaml
_cfg  = yaml.safe_load(open(CONFIG, encoding='utf-8'))
_mcfg = yaml.safe_load(open(_cfg['model_config'], encoding='utf-8'))
print(f"★ 라운드 {_cfg['run_name']} | base {_mcfg['hf_id']} | completion_only={_cfg.get('completion_only_loss', False)}")
print(f"  output_dir {_cfg['output_dir']} | 유효배치 {_cfg['per_device_train_batch_size']*_cfg['gradient_accumulation_steps']} (단일 GPU — DDP 분할 없음)")

## 4. 의존성 설치

`torch`가 `+cpu`로 뜨면 런타임 유형을 GPU로 변경 후 재시작.

In [ ]:
!pip install -q -e .[train]
!pip uninstall torchao -y -q
import os, torch
os.environ["WANDB_DISABLED"] = "true"
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available(), "ngpu:", torch.cuda.device_count())

## 5. 데이터 확인

In [ ]:
for p in ["data/processed/train.jsonl", "data/processed/val.jsonl", "data/eval/golden.jsonl"]:
    n = sum(1 for l in open(p, encoding="utf-8") if l.strip())
    print(f"{p}: {n} records")

## 6. 사전점검 — 학습 렌더 == 추론 프롬프트 + gold JSON

`OK` 확인 후 학습 진행. `FAIL`이면 model_config의 response_template 또는 chat_template 불일치.

In [ ]:
from transformers import AutoTokenizer
import json
_tok = AutoTokenizer.from_pretrained(_mcfg['hf_id'], trust_remote_code=_mcfg.get('trust_remote_code', False))
_user = "<채널: KakaoTalk>" + chr(10) + "<메시지>" + chr(10) + "내일 3시 회의" + chr(10) + "</메시지>"
_gold = json.dumps({'schedule_status': 'no', 'events': []}, ensure_ascii=False)
_sys  = _mcfg['system_prompt']
_full = [{'role':'system','content':_sys},{'role':'user','content':_user},{'role':'assistant','content':_gold}]
_pre  = [{'role':'system','content':_sys},{'role':'user','content':_user}]
_train = _tok.apply_chat_template(_full, tokenize=False, add_generation_prompt=False)
_extra = {}
if 'enable_thinking' in (getattr(_tok, 'chat_template', None) or ''):
    _extra['enable_thinking'] = False
_infer = _tok.apply_chat_template(_pre, tokenize=False, add_generation_prompt=True, **_extra)
_aligned = _train.startswith(_infer)
_json_next = (_train[len(_infer):] if _aligned else '').lstrip().startswith('{')
print('정렬:', _aligned, '| 추론 직후 JSON:', _json_next, '->', 'OK' if (_aligned and _json_next) else 'FAIL')
_rt = "<|im_start|>assistant" + chr(10)
print('completion-only response_template 존재:', _rt in _train, '(#1 마스킹 정상 전제)')
assert _aligned and _json_next, '학습/추론 프롬프트 정렬 깨짐'

## 7. 학습

단일 GPU · `python scripts/train_lora.py` 직접 실행 (torchrun 아님).  
첫 logging에서 loss가 감소하면 completion-only 마스킹 정상.

In [ ]:
print(f"학습 시작 → {CONFIG} (라운드: {_cfg['run_name']}, completion_only={_cfg.get('completion_only_loss', False)}) · 단일 GPU")
!python scripts/train_lora.py --config {CONFIG}

## 8. 평가 (LoRA merge → 골든셋 평가)

merge 후 `eval_model.py`로 골든 54건 채점. 결과는 `logs/eval_{라운드}.json`.

In [ ]:
import yaml, os
_cfg = yaml.safe_load(open(CONFIG, encoding='utf-8'))
_mcfg = yaml.safe_load(open(_cfg['model_config'], encoding='utf-8'))
BASE = _mcfg['hf_id']; LORA_DIR = _cfg['output_dir']; NAME = os.path.basename(LORA_DIR)
MERGED_DIR = f'models/merged/{NAME}'; EVAL_JSON = f'logs/eval_{NAME}.json'
GOLDEN = _cfg.get('eval_golden', 'data/eval/golden.jsonl'); ZIP = f'/content/lora_{NAME}.zip'
print('base', BASE); print('lora', LORA_DIR); print('golden', GOLDEN)
!python scripts/merge_lora.py --base {BASE} --lora {LORA_DIR} --out {MERGED_DIR}
!python scripts/eval_model.py --model {MERGED_DIR} --eval {GOLDEN} --out {EVAL_JSON} --model_config {_cfg['model_config']}
print(f'--- {EVAL_JSON} ---')
print(open(EVAL_JSON, encoding='utf-8').read())

## 9. 어댑터 다운로드

양자화(Q8_0)는 배포할 라운드만 **calendar_quantize.ipynb**에서 별도 수행.

In [ ]:
import shutil, os, glob
shutil.make_archive(ZIP[:-4], 'zip', LORA_DIR)
print('zip:', ZIP, os.path.getsize(ZIP)//1024//1024, 'MB')
from google.colab import files
files.download(ZIP)
files.download(EVAL_JSON)
for f in sorted(glob.glob('data/failures/round_latest*.jsonl')):
    print('failures:', f); files.download(f)